In [1]:
%matplotlib widget
import scipp as sc
import plopp as pp
import scippneutron as scn
import scippnexus as snx
import h5py
from pathlib import Path
import numpy as np

In [2]:
from ess.spectroscopy.indirect import bifrost
from bifrost2409.config import POOCH_DATA_DIR, INTERIM_DATA_DIR
from bifrost2409.dataset import download_datafiles

2024-11-19 14:15:50.714 | INFO     | bifrost2409.config:<module>:13 - PROJ_ROOT path is: /home/g/Projects/20240914/bifrost-workflow-2409


In [3]:
datafile = "20240914/BIFROST_20240914T053723.h5"
download_datafiles([datafile])

Fetching: 100%|██████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  7.12it/s]


In [4]:
targets = [
    'wavelength_monitor',
    'norm_events',
    'triplet_events',
]
target_files = {target: INTERIM_DATA_DIR / f'{Path(datafile).stem}_{target}.h5' for target in targets}
if all(file.exists() for file in target_files.values()):
    from scipp.io import load_hdf5
    from loguru import logger
    from rich.pretty import pretty_repr
    objects = {target: load_hdf5(file) for target, file in target_files.items()}
    logger.info(f'Loaded objects {pretty_repr(target_files)}')
else:
    data = bifrost(POOCH_DATA_DIR / datafile, is_simulated=True)
    objects = {target: data[target] for target in targets}
    for target in targets:
        objects[target].save_hdf5(target_files[target])

2024-11-19 14:15:58.149 | INFO     | __main__:<module>:12 - Loaded objects {
    'wavelength_monitor': PosixPath('/home/g/Projects/20240914/bifrost-workflow-2409/data/interim/BIFROST_20240914T053723_wavelength_monitor.h5'),
    'norm_events': PosixPath('/home/g/Projects/20240914/bifrost-workflow-2409/data/interim/BIFROST_20240914T053723_norm_events.h5'),
    'triplet_events': PosixPath('/home/g/Projects/20240914/bifrost-workflow-2409/data/interim/BIFROST_20240914T053723_triplet_events.h5')
}


In [5]:
trip = objects['triplet_events']

In [6]:
events, monitor = [objects[x] for x in ('norm_events', 'wavelength_monitor')]

In [7]:
from ess.spectroscopy.indirect import bifrost_to_nxspe
nxspe_output = INTERIM_DATA_DIR / 'nxspe' / f'{Path(datafile).stem}'
nxspe_files = bifrost_to_nxspe(events=events, output=nxspe_output)

100%|████████████████████████████████████████████████████████████████████████████████████| 180/180 [00:25<00:00,  6.94it/s]


In [8]:
# raise ValueError('ok')

In [9]:
# def hQxz(event_data, x_bins, z_bins, e_range):
#     x = event_data.bins.concat(event_data.dims)
#     x = x.bin(energy_transfer=e_range)[0]
#     x = x.bin(table_momentum_x=x_bins, table_momentum_z=x_bins)
#     return x.hist()

In [10]:
# hQxz(events, 200, 200, sc.array(values=[-0.01, 0.01], dims=['energy_transfer'], unit='meV')).plot()

In [11]:
from tqdm import tqdm
#del gr
for i in tqdm(range(monitor.sizes['setting']-2, monitor.sizes['setting'])):
    ev, mn = [x['setting', i] for x in (events, monitor)]
    # gr = ev.bin(incident_wavelength=mn.coords['incident_wavelength'])

100%|█████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 16677.15it/s]


In [12]:
from scipp import sqrt, scalar
from scipp.constants import Planck, neutron_mass

def lambda_to_ei(incident_wavelength):
    return ((Planck / incident_wavelength)**2 / neutron_mass / 2).to(unit='meV')

def ei_ef_to_en(incident_energy, final_energy):
    return incident_energy - final_energy

In [13]:
targets = ['energy_transfer', 'incident_energy', 'incident_wavelength']
graph = {
        'incident_energy': lambda_to_ei,
        'energy_transfer': ei_ef_to_en,
    }

In [14]:
def named_coords_midpoint_broadcast(data, names):
    from scipp import midpoints
    sizes = data.sizes
    def coord_midpoint_broadcast(coord):
        for x in coord.dims:
            if x in sizes and coord.sizes[x] == 1 + sizes[x]:
                coord = midpoints(coord, x)
        return coord.broadcast(sizes=sizes)
    
    return {k: coord_midpoint_broadcast(data.coords[k]) for k in names}

def get_empty_centres(data, keep: list[str], graph: dict, extract: list[str], dim: str):
    from scipp import array
    # histogram first
    data = data.hist().transform_coords(keep, graph=graph)
    empty = data.values == 0  # as used below, equivalent to numpy.nonzero(data.values == 0)
    coords = named_coords_midpoint_broadcast(data, extract)
    for k, v in coords.items():
        coords[k] = array(values=v.values[empty], dims=[dim], unit=v.unit, dtype=v.dtype)
        if v.variances is not None:
            coords[k].variances = v.variances[empty]
    return coords, empty


In [15]:
def initialize_needed(sizes, coord):
    from scipp import DType, full
    dtype = coord.dtype
    if dtype == DType.float64 or dtype == DType.float32:
        from numpy import nan
        default = nan
    elif dtype == DType.int32 or dtype == DType.int64:
        default = -1
    elif dtype == DType.string:
        default = ""
    elif dtype == DType.bool:
        default = False
    elif dtype == DType.datetime64:
        from scipp import datetime
        default = datetime(0)
    else:
        default = -1
    return full(sizes=sizes, unit=coord.unit, dtype=dtype, value=default)


In [16]:
def clean_up_observations(events, keep: list[str]):
    coords = [x for x in events.bins.coords if x not in keep]
    for coord in coords:
        del events.bins.coords[coord]
    return events

def add_null_observations(tevents, targets: list[str], graph: dict):
    # In the future it may be possible to do this without re-binning at the end
    # https://github.com/scipp/scipp/issues/1967#issuecomment-958680504
    from scipp import DataArray, zeros, concat, min as scipp_min, max as scipp_max
    from loguru import logger

    needed = tevents.dims
    orig = tevents.bins.constituents['data']
    dim = orig.dims[0]
    if any(n not in orig.coords for n in needed):
        bin_begin = tevents.bins.constituents['begin'].values.flatten()
        bin_end = tevents.bins.constituents['end'].values.flatten()
        for n in needed:
            t = initialize_needed(orig.sizes, tevents.coords[n])
            for v, b, e in zip(tevents.coords[n].values.flatten(), bin_begin, bin_end):
                t.values[b:e] = v
            orig.coords[n] = t

    extract = targets + [n for n in needed if n not in targets]
    coords, _ = get_empty_centres(tevents, targets, graph, extract, dim)
    rows = max(coords[target].sizes[dim] for target in targets)
    nulls = DataArray(zeros(sizes={dim: rows}, unit='counts', dtype=orig.dtype), coords=coords)
    if tevents.variances is not None:
        nulls.variances = 1 + nulls.values

    # extend the list of events
    comb = concat((orig, nulls), dim)

    # then bin or combine the coordinates of the input events
    binned = {k: tevents.coords[k] for k in extract if k in tevents.coords and tevents.coords[k].sizes[k] == 1 + tevents.sizes[k]}
    grouped = [k for k in extract if k in tevents.coords and k not in binned]
    # any other extract entries are left on the events' coordinates

    out = comb
    for group in grouped:
        smallest = scipp_min(tevents.coords[group])
        largest = scipp_max(tevents.coords[group])
        largest.value += 1  # label based slicing is upper-bound exclusive :(
        out = out.group(group)[group, smallest:largest]
    out = out.bin(binned)
    for k, v in tevents.coords.items():
        out.coords[k] = v
    return out

In [17]:
## max_int = sc.concat([sc.max(add_null_observations(clean_up_observations(events['setting', x], targets), targets, graph).hist()) for x in range(events.sizes['setting'])], 'setting')
#max_int = sc.concat([sc.max(clean_up_observations(events['setting', x], targets).hist()) for x in range(events.sizes['setting'])], 'setting')

In [18]:
#max_int.rename_dims(setting='a3').plot()

In [19]:
from scipp import DataArray
def _add_null_observations_append(events, targets: list[str], graph: dict):
    """Extend the events list after finding the number of empty bins. Then update the
    bin-indexing for the empty bins to point at the added end-of-list null events

    If done correctly, the input event information is undisturbed and the null events
    get assigned to bins which were otherwise unused.
    """
    from scipp import zeros, concat
    input_event_list = events.bins.constituents['data']
    needed = events.dims
    dim = input_event_list.dims[0]
    extract = targets + [n for n in needed if n not in targets]
    coords, empty = get_empty_centres(events, targets, graph, extract, dim)
    rows = max(coords[target].sizes[dim] for target in targets)
    nulls = DataArray(
        zeros(sizes={dim: rows}, unit='counts', dtype=input_event_list.dtype),
        coords=coords
    )
    if events.variances is not None:
        nulls.variances = 1 + nulls.values
    # make the new event list by concatenating the null observations on the end
    output_event_list = concat((input_event_list, nulls), dim=dim)
    # the first index for the _newly added_ null observations in the event list
    first = input_event_list.sizes[dim]
    return first, empty, nulls.sizes[dim]



In [20]:
cev = clean_up_observations(ev, targets)
fst, mt, nonull = _add_null_observations_append(cev, targets, graph)

In [21]:
np.cumsum(mt)

array([      1,       2,       3, ..., 1362601, 1362602, 1362603])

In [22]:
[x.shape for x in np.nonzero(mt)]

[(1362603,), (1362603,)]

In [23]:
nonull

1362603

In [24]:
averages = {t: cev.bins.coords[t].bins.nanmean() for t in targets}

In [25]:
list(averages)

['energy_transfer', 'incident_energy', 'incident_wavelength']

In [26]:
en = averages['energy_transfer']
en.plot()

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [27]:
hev = cev.hist().transform_coords(targets, graph=graph)
hev.coords['energy_transfer'].plot()

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [28]:
hev.coords['energy_transfer'].mean()

<scipp.Variable> ()    float64            [meV]  -0.0942

In [29]:
ev.bins.constituents['data']=None

In [30]:
ev

<scipp.DataArray>
Dimensions: Sizes[detector_number:13500, incident_wavelength:103, ]
Coordinates:
  a3                        float32            [deg]  ()  179
* a4                        float32            [deg]  ()  90
* detector_number             int64        <no unit>  (detector_number)  [1, 2, ..., 13499, 13500]
* final_energy              float64            [meV]  (detector_number)  [2.68475, 2.68449, ..., 5.06818, 5.06863]
* incident_wavelength       float64             [Å]  (incident_wavelength [bin-edge])  [3.85877, 3.868, ..., 5.673, 5.68182]
* monitor                   float64         [counts]  (incident_wavelength)  [1.70745e+06, 123158, ..., 7.54121e+06, 1.70707e+06]  [5.7189e+10, 3.25343e+09, ..., 3.72916e+11, 5.71634e+10]
* theta                     float64            [deg]  (detector_number)  [47.262, 47.3189, ..., 132.732, 132.787]
Data:
                          DataArrayView        <no unit>  (detector_number, incident_wavelength)  binned data: dim='event', content=DataArray(
          dims=(event: 11545264),
          data=float32[counts],
          coords={'energy_transfer':float64[meV], 'incident_energy':float64[meV],
                  'incident_wavelength':float64[Å]})

In [31]:
ev.bins.constituents['data']

<scipp.DataArray>
Dimensions: Sizes[event:11545264, ]
Coordinates:
* energy_transfer           float64            [meV]  (event)  [0.0147532, -0.00508892, ..., -0.00886787, -0.0421279]
* incident_energy           float64            [meV]  (event)  [2.69924, 2.6794, ..., 5.05887, 5.02561]
* incident_wavelength       float64             [Å]  (event)  [5.50512, 5.52547, ..., 4.02125, 4.03453]
Data:
                            float32         [counts]  (event)  [1, 1, ..., 1, 1]

In [32]:
c = sc.array(values=[-1, 1, 0.], dims=['event'], unit='counts')
x = sc.array(values=[0.1, 0.2, 2.3], dims=['event'], unit='m')
da = sc.DataArray(c, coords={'x': x})
bda = da.bin(x=sc.array(values=[0, 1, 2, 3.], dims=['x'], unit='m'))
bda

<scipp.DataArray>
Dimensions: Sizes[x:3, ]
Coordinates:
* x                         float64              [m]  (x [bin-edge])  [0, 1, 2, 3]
Data:
                          DataArrayView        <no unit>  (x)  binned data: dim='event', content=DataArray(
          dims=(event: 3),
          data=float64[counts],
          coords={'x':float64[m]})

In [33]:
bda.hist().values == 0

array([ True,  True,  True])

In [34]:
bda.bins.size().values == 0

array([False,  True, False])

In [35]:
events.bins